# ZPE 频率计算脚本

## 功能说明

1. **主程序模式**：
   - 扫描 ZPE 文件夹结构（1_body/ 和 2_body/）
   - 为每个 .vasp 或 POSCAR 文件创建频率计算任务目录
   - 生成 INCAR、KPOINTS，可选生成 POTCAR（需 vaspkit；无 vaspkit 时仅生成 POSCAR/INCAR/KPOINTS）
   - 创建批量提交脚本（job_list.txt 和 submit.sh）

2. **分析模式**：
   - 读取能量信息（TOTEN）
   - 读取振动频率信息
   - 计算零点能 (ZPE)
   - 检测虚频
   - 统计汇总并导出 CSV

3. **额外功能**：
   - 固定全部Cu原子
   - 数据分析（提取频率计算能量）
   - 数据清除（删除有问题的job文件夹）

In [17]:
# 导入库
import os
import shutil
import subprocess
import numpy as np
import matplotlib.pyplot as plt

In [18]:
# ============================================
# 主目录接口：修改这里即可
# ============================================
# ZPE 文件夹的根目录
# 方式1：扫描整个 ZPE（1_body + 2_body）
# zpe_root = "/home/ucaqzh0/thermol/100_water/absorption/absorption/ZPE"
# 方式2：仅扫描单个目录（如 H2O_u_2，该目录下需有 POSCAR 或 .vasp）
zpe_root = "/home/ucaqzh0/thermol/100_water/absorption/absorption/ZPE/high/"

In [20]:
# ============================
# INCAR TEMPLATE
# ============================
INCAR_template = """ISTART = 1

ENCUT = 450
PREC = ACCURATE
LREAL = .FALSE.
ISMEAR = 1
SIGMA  = 0.2
GGA    = PE
NELM   = 120
NELMIN = 8
NELMDL = -5
EDIFF  = 1E-7
EDIFFG = -0.01
NSW    = 1
ISIF   = 2
IBRION = 5
POTIM  = 0.01
NFREE  = 2
NWRITE = 3

LWAVE  = .FALSE.
LCHARG = .FALSE.

ISPIN = 1
LORBIT = 11
IVDW = 12
IDIPOL = 3
LDIPOL = .TRUE.
NCORE = 8
ISYM = 0
IALGO = 48
"""

# ============================
# KPOINTS TEMPLATE
# ============================
KPOINTS_template = """Automatic mesh
0
Gamma
4 4 1
0 0 0
"""

# ===================================
# SUBMIT.SH HEADER（Young 配置）
# ===================================
submit_header_tpl = """#!/bin/bash
# stack under SGE with Intel MPI.

#$ -N {job_name}
#$ -S /bin/bash
#$ -l h_rt=2:00:00
#$ -P Gold
#$ -A UCL_chemM_Catlow
#$ -pe mpi 128
#$ -cwd

source /etc/profile.d/modules.sh
module load vasp/5.4.4-18apr2017-vtst-r178/intel-2017-update1

"""


In [21]:
# =========================================================
# 辅助函数
# =========================================================
import os
import re

def vasp_filename_to_job_name(vasp_filename):
    """从 .vasp 文件名得到提交用的任务名称（无后缀、可含路径）。"""
    return os.path.splitext(os.path.basename(vasp_filename))[0]


def read_energy(path):
    """读取能量"""
    osz = os.path.join(path, "OSZICAR")
    if os.path.exists(osz):
        with open(osz) as f:
            for line in reversed(f.readlines()):
                if "E0=" in line:
                    return float(line.split()[2])

    outc = os.path.join(path, "OUTCAR")
    if os.path.exists(outc):
        with open(outc) as f:
            for line in f:
                if "free  energy   TOTEN" in line:
                    return float(line.split()[4])

    return None


def scan_zpe_structure(zpe_root):
    """扫描 ZPE 文件夹，返回所有 .vasp 或 POSCAR 文件的路径和对应的 job 目录路径
    
    支持两种目录结构：
    1. 标准结构：zpe_root 下含 1_body/ 和 2_body/ 子目录
    2. 单目录模式：zpe_root 直接包含 .vasp 或 POSCAR 文件（如 2_body/H2O_u_2）
    """
    structure_files = []
    job_dirs = []
    
    # 检查是否为标准结构（含 1_body 或 2_body）
    has_standard = any(os.path.exists(os.path.join(zpe_root, b)) for b in ["1_body", "2_body"])
    
    if has_standard:
        # 扫描 1_body 和 2_body
        for body_type in ["1_body", "2_body"]:
            body_path = os.path.join(zpe_root, body_type)
            if not os.path.exists(body_path):
                continue
            
            for molecule_dir in os.listdir(body_path):
                molecule_path = os.path.join(body_path, molecule_dir)
                if not os.path.isdir(molecule_path):
                    continue
                
                for file in os.listdir(molecule_path):
                    if file.endswith(".vasp"):
                        struct_path = os.path.join(molecule_path, file)
                        structure_files.append(struct_path)
                        basename = os.path.splitext(file)[0]
                        job_dir = os.path.join(molecule_path, basename)
                        job_dirs.append(job_dir)
                    elif file == "POSCAR":
                        struct_path = os.path.join(molecule_path, file)
                        structure_files.append(struct_path)
                        job_dir = os.path.join(molecule_path, "job_POSCAR")
                        job_dirs.append(job_dir)
    else:
        # 单目录模式：直接在 zpe_root 下查找 .vasp 或 POSCAR
        for file in os.listdir(zpe_root):
            if file.endswith(".vasp"):
                struct_path = os.path.join(zpe_root, file)
                structure_files.append(struct_path)
                basename = os.path.splitext(file)[0]
                job_dir = os.path.join(zpe_root, basename)
                job_dirs.append(job_dir)
            elif file == "POSCAR":
                struct_path = os.path.join(zpe_root, file)
                structure_files.append(struct_path)
                job_dir = os.path.join(zpe_root, "job_POSCAR")
                job_dirs.append(job_dir)
    
    return structure_files, job_dirs


def read_frequencies(outcar_path):
    """
    从 OUTCAR 读取频率信息（包括全部频率和虚频），并计算ZPE
    返回: {
        'frequencies': [全部频率列表, cm^-1]（包括实频和虚频的绝对值，用于ZPE计算）,
        'zpe': 零点能 (eV)（从频率计算得出）,
        'imaginary_freqs': [虚频列表, cm^-1]（负值）,
        'real_freqs': [实频列表, cm^-1]（正值）,
        'n_modes': 振动模式数量
    }
    """
    if not os.path.exists(outcar_path):
        return None
    
    all_frequencies = []  # 全部频率（用于ZPE计算，包括虚频的绝对值）
    imaginary_freqs = []  # 虚频（负值）
    real_freqs = []       # 实频（正值）
    
    with open(outcar_path, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()
        
        # 使用正则表达式提取所有频率，更可靠
        # 匹配格式：数字 cm^-1 或 数字 cm-1（可能带负号，支持科学计数法）
        # 匹配模式：可选负号 + 数字 + 可选小数点 + 可选科学计数法 + 空格 + cm-1 或 cm^-1
        freq_pattern = r'(-?\d+\.?\d*(?:[eE][+-]?\d+)?)\s+cm-1'
        
        # 方法1: 查找包含THz和cm^-1/cm-1的行（主要方法）
        for i, line in enumerate(lines):
            if ("THz" in line and ("cm^-1" in line or "cm-1" in line)):
                # 使用正则表达式提取所有频率值（可能一行有多个）
                matches = re.findall(freq_pattern, line)
                for match in matches:
                    try:
                        freq_cm = float(match)
                        
                        # 判断是否为虚频：
                        # 1. 负值
                        # 2. 行中包含 "f/i" 或第二个字段是 "i"
                        line_parts = line.split()
                        is_imaginary = (freq_cm < 0) or ("f/i" in line) or (len(line_parts) > 1 and line_parts[1] == "i")
                        
                        if is_imaginary:
                            imaginary_freqs.append(freq_cm)
                            # ZPE计算使用绝对值
                            all_frequencies.append(abs(freq_cm))
                        else:
                            real_freqs.append(freq_cm)
                            all_frequencies.append(freq_cm)
                    except (ValueError, IndexError):
                        pass
        
        # 方法2: 如果方法1没找到，尝试查找其他可能的频率格式
        if len(all_frequencies) == 0:
            # 查找 "Eigenvalues" 或 "frequencies" 相关的行
            for i, line in enumerate(lines):
                if ("Eigenvalues" in line or "frequencies" in line.lower() or "vibrational" in line.lower()):
                    # 查找接下来的几行中的频率
                    for j in range(i+1, min(i+100, len(lines))):
                        freq_line = lines[j]
                        if ("THz" in freq_line and ("cm^-1" in freq_line or "cm-1" in freq_line)):
                            matches = re.findall(freq_pattern, freq_line)
                            for match in matches:
                                try:
                                    freq_cm = float(match)
                                    line_parts = freq_line.split()
                                    is_imaginary = (freq_cm < 0) or ("f/i" in freq_line) or (len(line_parts) > 1 and line_parts[1] == "i")
                                    
                                    if is_imaginary:
                                        imaginary_freqs.append(freq_cm)
                                        all_frequencies.append(abs(freq_cm))
                                    else:
                                        real_freqs.append(freq_cm)
                                        all_frequencies.append(freq_cm)
                                except (ValueError, IndexError):
                                    pass
                        # 如果找到频率行，继续查找直到遇到空行或新章节
                        if matches and j > i+10:
                            break
    
    # 去重：确保每个频率只计算一次（保留顺序）
    # 使用字典来跟踪已见过的频率（保留第一个出现的）
    seen_all = {}
    seen_real = {}
    seen_imaginary = {}
    
    unique_all_freqs = []
    unique_real_freqs = []
    unique_imaginary_freqs = []
    
    for freq in all_frequencies:
        # 使用精度0.01 cm^-1来判断是否重复
        freq_rounded = round(freq, 2)
        if freq_rounded not in seen_all:
            seen_all[freq_rounded] = freq
            unique_all_freqs.append(freq)
    
    for freq in real_freqs:
        freq_rounded = round(freq, 2)
        if freq_rounded not in seen_real:
            seen_real[freq_rounded] = freq
            unique_real_freqs.append(freq)
    
    for freq in imaginary_freqs:
        freq_rounded = round(freq, 2)
        if freq_rounded not in seen_imaginary:
            seen_imaginary[freq_rounded] = freq
            unique_imaginary_freqs.append(freq)
    
    # 更新列表
    all_frequencies = unique_all_freqs
    real_freqs = unique_real_freqs
    imaginary_freqs = unique_imaginary_freqs
    
    # 计算零点能 (ZPE) - 从全部频率计算（包括虚频的绝对值）
    # 使用简化公式: ZPE (eV) = Σ(频率, cm^-1) / 2000
    # 这个公式等价于: ZPE = 0.5 * h * c * Σ(ν) / (h * c / 2000)
    # 其中 h = 4.135667696e-15 eV·s, c = 2.99792458e10 cm/s
    zpe = None
    if all_frequencies:
        # 使用简化公式：ZPE = Σ(频率) / 2000
        zpe = sum(all_frequencies) / 2000.0
    
    return {
        'frequencies': all_frequencies,  # 全部频率（用于显示和ZPE计算）
        'zpe': zpe,                      # 零点能（从频率计算得出）
        'imaginary_freqs': imaginary_freqs,  # 虚频列表（负值）
        'real_freqs': real_freqs,         # 实频列表（正值）
        'n_modes': len(all_frequencies)   # 总模式数
    }

## 主程序：创建任务目录

**注意**：请先运行上方的 Cell 4（辅助函数），再运行下方主程序。

In [22]:
# 主程序：创建频率计算任务目录
# 检查依赖：需先运行 Cell 4 定义 scan_zpe_structure
if 'scan_zpe_structure' not in globals():
    raise RuntimeError("请先运行 Cell 4（辅助函数）以定义 scan_zpe_structure，再运行本单元")

print(f"📌 正在扫描 ZPE 目录结构：{zpe_root}")

# 扫描所有 .vasp 或 POSCAR 文件
structure_files, job_dirs = scan_zpe_structure(zpe_root)

if len(structure_files) == 0:
    print("❌ 未找到 .vasp 或 POSCAR 文件")
else:
    print(f"\n找到 {len(structure_files)} 个结构文件：")
    for struct_file in structure_files:
        rel_path = os.path.relpath(struct_file, zpe_root)
        print(f"  - {rel_path}")
    
    print(f"\n{'='*60}")
    print("开始创建任务目录...")
    print(f"{'='*60}\n")
    
    created_jobs = []
    
    for struct_file, job_dir in zip(structure_files, job_dirs):
        # 检查 job 目录是否已存在且包含必要文件
        if os.path.exists(job_dir) and os.path.exists(os.path.join(job_dir, "INCAR")):
            print(f"⏭  跳过（已存在）: {os.path.relpath(job_dir, zpe_root)}")
            created_jobs.append(job_dir)
            continue
        
        print(f"\n>>> 创建任务目录: {os.path.relpath(job_dir, zpe_root)}")
        
        # 创建 job 目录
        os.makedirs(job_dir, exist_ok=True)
        
        # 复制结构文件（.vasp 或 POSCAR）为 job 目录下的 POSCAR
        shutil.copy(struct_file, os.path.join(job_dir, "POSCAR"))
        
        # 写入 INCAR
        incar_path = os.path.join(job_dir, "INCAR")
        with open(incar_path, "w") as f:
            f.write(INCAR_template)
        
        # 写入 KPOINTS
        kpoints_path = os.path.join(job_dir, "KPOINTS")
        with open(kpoints_path, "w") as f:
            f.write(KPOINTS_template)
        
        # 生成 POTCAR（可选：无 vaspkit 时跳过，仅生成 POSCAR/INCAR/KPOINTS）
        try:
            result = subprocess.run(
                "vaspkit -task 103",
                shell=True,
                cwd=job_dir,
                capture_output=True,
                text=True,
                timeout=30
            )
            if result.returncode != 0:
                print(f"  ⚠  vaspkit 执行警告，跳过 POTCAR: {result.stderr[:80] if result.stderr else '未知错误'}")
            else:
                print(f"  ✔  POTCAR 已生成")
        except (FileNotFoundError, subprocess.TimeoutExpired) as e:
            print(f"  ⏭  跳过 POTCAR（无 vaspkit 或超时），仅生成 POSCAR/INCAR/KPOINTS")
        
        created_jobs.append(job_dir)
    
    # ============================
    # 写 job_list.txt 和 submit.sh（使用绝对路径）
    # ============================
    print(f"\n{'='*60}")
    print("生成提交脚本...")
    print(f"{'='*60}\n")
    
    job_list_path = os.path.join(zpe_root, "job_list.txt")
    with open(job_list_path, "w") as f:
        for d in created_jobs:
            f.write(os.path.abspath(d) + "\n")
    
    print(f"✔ job_list.txt 已生成: {job_list_path}")
    print(f"  包含 {len(created_jobs)} 个任务")
    
    # 生成 submit.sh
    job_name = "ZPE_frequency"
    submit_path = os.path.join(zpe_root, "submit.sh")
    with open(submit_path, "w") as f:
        f.write(submit_header_tpl.format(job_name=job_name))
        f.write(f"\n#$ -t 1-{len(created_jobs)}\n")
        
        f.write(f"""
JOB_LIST="{os.path.abspath(job_list_path)}"
JOB_DIR=$(sed -n "${{SGE_TASK_ID}}p" "$JOB_LIST")
JOB_NAME=$(basename "$JOB_DIR")

echo "========== Running Frequency Calculation: $JOB_NAME =========="
echo "Task ID: $SGE_TASK_ID"
echo "Job Directory: $JOB_DIR"
cd "$JOB_DIR"
gerun vasp_std > "$JOB_DIR/result.$JOB_ID.$SGE_TASK_ID.$JOB_NAME"
echo "========== Completed: $JOB_NAME =========="
""")
    
    print(f"✔ submit.sh 已生成: {submit_path}")
    print(f"\n>>> 提交任务命令：")
    print(f"   cd {zpe_root}")
    print(f"   qsub submit.sh")
    print(f"\n💡 提示：脚本已准备好，请手动执行 qsub 命令提交任务")

📌 正在扫描 ZPE 目录结构：/home/ucaqzh0/thermol/100_water/absorption/absorption/ZPE/high/

找到 2 个结构文件：
  - POSCAR_H2_02.vasp
  - POSCAR_OH_dim.vasp

开始创建任务目录...


>>> 创建任务目录: POSCAR_H2_02
  ✔  POTCAR 已生成

>>> 创建任务目录: POSCAR_OH_dim
  ✔  POTCAR 已生成

生成提交脚本...

✔ job_list.txt 已生成: /home/ucaqzh0/thermol/100_water/absorption/absorption/ZPE/high/job_list.txt
  包含 2 个任务
✔ submit.sh 已生成: /home/ucaqzh0/thermol/100_water/absorption/absorption/ZPE/high/submit.sh

>>> 提交任务命令：
   cd /home/ucaqzh0/thermol/100_water/absorption/absorption/ZPE/high/
   qsub submit.sh

💡 提示：脚本已准备好，请手动执行 qsub 命令提交任务


## 功能1：固定全部Cu原子

此功能会修改所有job目录中的POSCAR文件，将所有Cu原子的坐标固定（设置为 F F F）。

In [23]:
# 功能1：固定全部Cu原子
def fix_all_cu_atoms(zpe_root):
    """
    修改所有job目录中的POSCAR文件，将所有Cu原子固定（设置为 F F F）
    """
    _, job_dirs = scan_zpe_structure(zpe_root)
    
    fixed_count = 0
    skipped_count = 0
    
    for job_dir in job_dirs:
        poscar_path = os.path.join(job_dir, "POSCAR")
        if not os.path.exists(poscar_path):
            print(f"⚠ 跳过（不存在POSCAR）: {os.path.relpath(job_dir, zpe_root)}")
            skipped_count += 1
            continue
        
        # 读取POSCAR文件
        with open(poscar_path, 'r') as f:
            lines = f.readlines()
        
        # 查找元素类型行（第6行，索引5）
        if len(lines) < 6:
            print(f"⚠ 跳过（格式错误）: {os.path.relpath(job_dir, zpe_root)}")
            skipped_count += 1
            continue
        
        # 解析元素类型和数量
        element_line = lines[5].strip().split()
        num_line = lines[6].strip().split()
        
        if len(element_line) == 0 or len(num_line) == 0:
            print(f"⚠ 跳过（无法解析元素）: {os.path.relpath(job_dir, zpe_root)}")
            skipped_count += 1
            continue
        
        # 检查是否有Selective dynamics
        has_selective = False
        selective_idx = None
        coord_start_idx = None
        
        for i, line in enumerate(lines):
            if "Selective dynamics" in line:
                has_selective = True
                selective_idx = i
                coord_start_idx = i + 2  # 坐标从Selective dynamics后两行开始
                break
        
        if not has_selective:
            print(f"⚠ 跳过（无Selective dynamics）: {os.path.relpath(job_dir, zpe_root)}")
            skipped_count += 1
            continue
        
        # 找到Cu原子的数量（第一个元素）
        cu_count = 0
        if element_line[0].upper() == 'CU' or element_line[0] == 'Cu':
            try:
                cu_count = int(num_line[0])
            except (ValueError, IndexError):
                print(f"⚠ 跳过（无法解析Cu数量）: {os.path.relpath(job_dir, zpe_root)}")
                skipped_count += 1
                continue
        
        if cu_count == 0:
            print(f"⚠ 跳过（无Cu原子）: {os.path.relpath(job_dir, zpe_root)}")
            skipped_count += 1
            continue
        
        # 修改Cu原子的固定状态
        modified = False
        atom_idx = 0
        
        for i in range(coord_start_idx, len(lines)):
            line = lines[i].strip()
            if not line or len(line.split()) < 3:
                break
            
            if atom_idx < cu_count:
                parts = line.split()
                if len(parts) >= 6:
                    # 修改为 F F F
                    new_line = f"{parts[0]:>20} {parts[1]:>20} {parts[2]:>20}   F   F   F\n"
                    lines[i] = new_line
                    modified = True
                atom_idx += 1
            else:
                break
        
        if modified:
            # 写回文件
            with open(poscar_path, 'w') as f:
                f.writelines(lines)
            print(f"✔ 已固定 {cu_count} 个Cu原子: {os.path.relpath(job_dir, zpe_root)}")
            fixed_count += 1
        else:
            print(f"⚠ 未修改: {os.path.relpath(job_dir, zpe_root)}")
            skipped_count += 1
    
    print(f"\n{'='*60}")
    print(f"完成！")
    print(f"  已修改: {fixed_count} 个job目录")
    print(f"  已跳过: {skipped_count} 个job目录")
    print(f"{'='*60}")

# 执行固定Cu原子操作
fix_all_cu_atoms(zpe_root)

✔ 已固定 48 个Cu原子: POSCAR_H2_02
✔ 已固定 48 个Cu原子: POSCAR_OH_dim

完成！
  已修改: 2 个job目录
  已跳过: 0 个job目录


## 功能2：数据分析 - 提取频率计算能量

此功能分析所有已完成的频率计算结果，提取能量和频率信息，并保存到CSV文件。

In [16]:
# 功能2：数据分析 - 提取频率计算能量
def analyze_results():
    """分析所有已完成的频率计算结果"""
    print("📊 分析模式：读取频率计算结果...")
    print("="*80)
    
    _, job_dirs = scan_zpe_structure(zpe_root)
    
    results = []
    summary = {
        '1_body': {},
        '2_body': {}
    }
    
    for job_dir in job_dirs:
        outcar_path = os.path.join(job_dir, "OUTCAR")
        if not os.path.exists(outcar_path):
            continue
        
        energy = read_energy(job_dir)
        freq_data = read_frequencies(outcar_path)
        rel_path = os.path.relpath(job_dir, zpe_root)
        
        # 判断是 1_body 还是 2_body
        body_type = "1_body" if "1_body" in rel_path else "2_body"
        molecule = rel_path.split("/")[1] if "/" in rel_path else "unknown"
        
        result = {
            "path": rel_path,
            "energy": energy,
            "freq_data": freq_data,
            "body_type": body_type,
            "molecule": molecule
        }
        results.append(result)
        
        # 打印详细信息
        print(f"\n📁 {rel_path}")
        
        if freq_data:
            # 显示全部频率信息
            if freq_data['frequencies']:
                print(f"   总振动模式数: {freq_data['n_modes']}")
                print(f"   全部频率 ({len(freq_data['frequencies'])}个): {[f'{f:.2f}' for f in freq_data['frequencies']]} cm^-1")
                
                if freq_data.get('real_freqs'):
                    print(f"   实频数量: {len(freq_data['real_freqs'])}")
                    if len(freq_data['real_freqs']) > 0:
                        print(f"   实频范围: {min(freq_data['real_freqs']):.2f} - {max(freq_data['real_freqs']):.2f} cm^-1")
            
            if freq_data['imaginary_freqs']:
                print(f"   ⚠ 虚频数量: {len(freq_data['imaginary_freqs'])}")
                print(f"   虚频: {[f'{f:.2f}' for f in freq_data['imaginary_freqs']]} cm^-1")
            
            # 显示ZPE计算结果和公式
            if freq_data['zpe'] is not None:
                print(f"\n   零点能 (ZPE) 计算:")
                print(f"   公式: ZPE = Σ(频率, cm^-1) / 2000")
                print(f"   Σ(频率) = {sum(freq_data['frequencies']):.6f} cm^-1")
                print(f"   ZPE = {sum(freq_data['frequencies']):.6f} / 2000")
                print(f"   ZPE = {freq_data['zpe']:.6f} eV")
            else:
                print(f"   ⚠ 无法计算ZPE（无频率数据）")
            
            # 存储到汇总
            if molecule not in summary[body_type]:
                summary[body_type][molecule] = []
            summary[body_type][molecule].append({
                'zpe': freq_data['zpe'],
                'n_freqs': len(freq_data['frequencies']),
                'n_real_freqs': len(freq_data.get('real_freqs', [])),
                'n_imaginary': len(freq_data['imaginary_freqs'])
            })
        else:
            print(f"   ⚠ 无频率数据（可能计算未完成）")
    
    # 打印汇总
    print(f"\n{'='*80}")
    print("📊 汇总统计")
    print(f"{'='*80}")
    
    for body_type in ["1_body", "2_body"]:
        if summary[body_type]:
            print(f"\n{body_type}:")
            for molecule, mol_results in summary[body_type].items():
                completed = [r for r in mol_results if r['zpe'] is not None]
                with_imaginary = [r for r in completed if r['n_imaginary'] > 0]
                
                print(f"  {molecule}:")
                print(f"    已完成: {len(completed)}/{len(mol_results)}")
                if completed:
                    avg_zpe = np.mean([r['zpe'] for r in completed])
                    print(f"    平均 ZPE: {avg_zpe:.6f} eV")
                if with_imaginary:
                    print(f"    ⚠ {len(with_imaginary)} 个结构存在虚频")
    
    print(f"\n✔ 总共找到 {len(results)} 个已完成的计算")
    
    # 保存结果到 CSV
    csv_path = os.path.join(zpe_root, "frequency_results.csv")
    try:
        import pandas as pd
        df_data = []
        for r in results:
            if r['freq_data'] and r['freq_data']['zpe'] is not None:
                # 将频率列表转换为字符串，用分号分隔
                all_freqs_str = ';'.join([f'{f:.6f}' for f in r['freq_data']['frequencies']]) if r['freq_data']['frequencies'] else ''
                real_freqs_str = ';'.join([f'{f:.6f}' for f in r['freq_data'].get('real_freqs', [])]) if r['freq_data'].get('real_freqs') else ''
                imaginary_freqs_str = ';'.join([f'{f:.6f}' for f in r['freq_data']['imaginary_freqs']]) if r['freq_data']['imaginary_freqs'] else ''
                
                df_data.append({
                    'path': r['path'],
                    'body_type': r['body_type'],
                    'molecule': r['molecule'],
                    'zpe_eV': r['freq_data']['zpe'],
                    'zpe_formula': 'ZPE = Σ(频率, cm^-1) / 2000',
                    'sum_frequencies_cm-1': sum(r['freq_data']['frequencies']) if r['freq_data']['frequencies'] else 0,
                    'n_frequencies': len(r['freq_data']['frequencies']),
                    'n_real_frequencies': len(r['freq_data'].get('real_freqs', [])),
                    'n_imaginary': len(r['freq_data']['imaginary_freqs']),
                    'all_frequencies_cm-1': all_freqs_str,  # 全部频率列表
                    'real_frequencies_cm-1': real_freqs_str,  # 实频列表
                    'imaginary_frequencies_cm-1': imaginary_freqs_str,  # 虚频列表
                    'min_freq_cm-1': min(r['freq_data']['frequencies']) if r['freq_data']['frequencies'] else None,
                    'max_freq_cm-1': max(r['freq_data']['frequencies']) if r['freq_data']['frequencies'] else None,
                })
        
        if df_data:
            df = pd.DataFrame(df_data)
            df.to_csv(csv_path, index=False)
            print(f"✔ 结果已保存到: {csv_path}")
    except ImportError:
        print("💡 提示: 安装 pandas 可自动保存结果到 CSV")
    
    return results

# 执行数据分析
results = analyze_results()

📊 分析模式：读取频率计算结果...

📁 POSCAR_H2
   总振动模式数: 6
   全部频率 (6个): ['1730.21', '1637.67', '129.33', '347.10', '354.16', '1081.50'] cm^-1
   实频数量: 2
   实频范围: 1637.67 - 1730.21 cm^-1
   ⚠ 虚频数量: 4
   虚频: ['129.33', '347.10', '354.16', '1081.50'] cm^-1

   零点能 (ZPE) 计算:
   公式: ZPE = Σ(频率, cm^-1) / 2000
   Σ(频率) = 5279.980465 cm^-1
   ZPE = 5279.980465 / 2000
   ZPE = 2.639990 eV

📁 POSCAR_OH_dim
   总振动模式数: 6
   全部频率 (6个): ['994.24', '825.47', '363.86', '323.72', '312.63', '237.72'] cm^-1
   实频数量: 6
   实频范围: 237.72 - 994.24 cm^-1

   零点能 (ZPE) 计算:
   公式: ZPE = Σ(频率, cm^-1) / 2000
   Σ(频率) = 3057.639465 cm^-1
   ZPE = 3057.639465 / 2000
   ZPE = 1.528820 eV

📁 POSCAR_H2O
   总振动模式数: 9
   全部频率 (9个): ['3597.90', '886.71', '700.71', '536.64', '507.41', '328.89', '204.33', '198.96', '1179.02'] cm^-1
   实频数量: 8
   实频范围: 198.96 - 3597.90 cm^-1
   ⚠ 虚频数量: 1
   虚频: ['1179.02'] cm^-1

   零点能 (ZPE) 计算:
   公式: ZPE = Σ(频率, cm^-1) / 2000
   Σ(频率) = 8140.569573 cm^-1
   ZPE = 8140.569573 / 2000
   ZPE = 4.070285 e

## 功能3：数据清除 - 删除有问题的job文件夹

此功能可以删除指定的job文件夹，用于清理计算出现问题的任务。可以删除单个文件夹或批量删除。

In [9]:
# =========================================================
# 设置root路径（只作用于这个cell，修改这里即可）
# =========================================================
root = "/home/ucaqzh0/thermol/100_water/absorption/absorption/ZPE"  # 修改为你的路径

# 功能3：数据清除 - 删除有问题的job文件夹
def clean_job_directories(root, confirm=True):
    if not os.path.exists(root):
        print(f"❌ 错误：目录不存在: {root}")
        return
    
    root = os.path.abspath(root)
    to_delete = []
    
    print(f"📋 扫描根目录: {root}")
    print("="*80)
    
    # 判断root的层级
    root_basename = os.path.basename(root)
    root_parent = os.path.dirname(root)
    root_parent_basename = os.path.basename(root_parent)
    
    # 情况1：root是ZPE级别（包含1_body和2_body）
    if os.path.exists(os.path.join(root, "1_body")) or os.path.exists(os.path.join(root, "2_body")):
        print("📂 检测到ZPE级别，扫描1_body和2_body...")
        for body_type in ["1_body", "2_body"]:
            body_path = os.path.join(root, body_type)
            if os.path.exists(body_path):
                # 扫描该body下的所有分子目录
                for molecule_dir in os.listdir(body_path):
                    molecule_path = os.path.join(body_path, molecule_dir)
                    if os.path.isdir(molecule_path):
                        # 扫描该分子目录下的所有job目录
                        for item in os.listdir(molecule_path):
                            job_path = os.path.join(molecule_path, item)
                            if os.path.isdir(job_path) and item.startswith("job_"):
                                rel_path = os.path.relpath(job_path, root)
                                to_delete.append((job_path, rel_path))
    
    # 情况2：root是1_body或2_body级别
    elif root_basename in ["1_body", "2_body"]:
        print(f"📂 检测到{root_basename}级别，扫描所有子文件夹...")
        # 扫描该body下的所有分子目录
        for molecule_dir in os.listdir(root):
            molecule_path = os.path.join(root, molecule_dir)
            if os.path.isdir(molecule_path):
                # 扫描该分子目录下的所有job目录
                for item in os.listdir(molecule_path):
                    job_path = os.path.join(molecule_path, item)
                    if os.path.isdir(job_path) and item.startswith("job_"):
                        rel_path = os.path.relpath(job_path, root)
                        to_delete.append((job_path, rel_path))
    
    # 情况3：root是分子级别（如H、H2）
    # 检查父目录是否是1_body或2_body，且当前目录下有job_开头的目录
    elif os.path.basename(root_parent) in ["1_body", "2_body"]:
        # 检查是否有job_开头的目录
        has_job_dirs = any(item.startswith("job_") and os.path.isdir(os.path.join(root, item)) 
                          for item in os.listdir(root))
        if has_job_dirs:
            print(f"📂 检测到分子级别，扫描该目录下的所有job目录...")
            # 扫描该分子目录下的所有job目录
            for item in os.listdir(root):
                job_path = os.path.join(root, item)
                if os.path.isdir(job_path) and item.startswith("job_"):
                    rel_path = os.path.relpath(job_path, root)
                    to_delete.append((job_path, rel_path))
        else:
            # 如果没有job目录，可能是其他层级，继续到情况5
            print(f"📂 检测到分子级别但无job目录，尝试递归查找...")
            for root_dir, dirs, files in os.walk(root):
                for dir_name in dirs:
                    if dir_name.startswith("job_"):
                        job_path = os.path.join(root_dir, dir_name)
                        rel_path = os.path.relpath(job_path, root)
                        to_delete.append((job_path, rel_path))
    
    # 情况4：root是job目录级别
    elif root_basename.startswith("job_"):
        print(f"📂 检测到job目录级别，直接删除该目录...")
        rel_path = os.path.relpath(root, os.path.dirname(os.path.dirname(root)))
        to_delete.append((root, rel_path))
    
    # 情况5：其他情况，尝试扫描所有以job_开头的子目录
    else:
        print(f"📂 未识别层级，尝试扫描所有job_开头的目录...")
        for item in os.listdir(root):
            item_path = os.path.join(root, item)
            if os.path.isdir(item_path) and item.startswith("job_"):
                rel_path = os.path.relpath(item_path, root)
                to_delete.append((item_path, rel_path))
        
        # 如果没有找到job目录，尝试递归查找
        if len(to_delete) == 0:
            print("   未找到job目录，尝试递归查找...")
            for root_dir, dirs, files in os.walk(root):
                for dir_name in dirs:
                    if dir_name.startswith("job_"):
                        job_path = os.path.join(root_dir, dir_name)
                        rel_path = os.path.relpath(job_path, root)
                        to_delete.append((job_path, rel_path))
    
    if len(to_delete) == 0:
        print("❌ 未找到任何job目录")
        return
    
    # 显示将要删除的文件夹
    print(f"\n📋 找到 {len(to_delete)} 个job目录：")
    print("="*80)
    for i, (job_dir, rel_path) in enumerate(to_delete, 1):
        exists = "存在" if os.path.exists(job_dir) else "不存在"
        print(f"{i:3d}. {rel_path} ({exists})")
    print("="*80)
    
    # 确认删除
    if confirm:
        response = input("\n⚠ 确认删除以上所有job目录？(yes/no): ").strip().lower()
        if response not in ['yes', 'y']:
            print("❌ 已取消删除操作")
            return
    
    # 执行删除
    deleted_count = 0
    failed_count = 0
    
    for job_dir, rel_path in to_delete:
        try:
            if os.path.exists(job_dir):
                shutil.rmtree(job_dir)
                print(f"✔ 已删除: {rel_path}")
                deleted_count += 1
            else:
                print(f"⚠ 目录不存在: {rel_path}")
        except Exception as e:
            print(f"❌ 删除失败: {rel_path} - {str(e)}")
            failed_count += 1
    
    print(f"\n{'='*60}")
    print(f"完成！")
    print(f"  已删除: {deleted_count} 个目录")
    print(f"  失败: {failed_count} 个目录")
    print(f"{'='*60}")

# =========================================================
# 使用说明
# 
# 目录结构说明：
# ZPE/ (根目录)
#   ├── 1_body/
#   │   ├── H/
#   │   │   └── job_POSCAR_H_hollow/
#   │   ├── H2/
#   │   │   └── job_POSCAR_H2_top/
#   │   └── ...
#   └── 2_body/
#       ├── H/
#       ├── H_O/
#       └── ...
# 
# 函数会自动根据给定的root判断层级并扫描相应的job目录
# =========================================================

# 示例1：扫描ZPE下的所有job目录（1_body和2_body）
clean_job_directories(root)

# 示例2：扫描1_body下的所有job目录
# clean_job_directories(os.path.join(root, '1_body'))

# 示例3：扫描2_body下的所有job目录
# clean_job_directories(os.path.join(root, '2_body'))

# 示例4：扫描H2分子下的所有job目录
# clean_job_directories(os.path.join(root, '1_body', 'H2'))

# 示例5：直接删除某个job目录
# clean_job_directories(os.path.join(root, '1_body', 'H2', 'job_POSCAR_H2_top'))

# 示例6：不确认直接删除（谨慎使用！）
# clean_job_directories(root, confirm=False)



📋 扫描根目录: /home/ucaqzh0/thermol/100_water/absorption/absorption/ZPE
📂 检测到ZPE级别，扫描1_body和2_body...

📋 找到 9 个job目录：
  1. 1_body/H2/job_POSCAR_H2_top (存在)
  2. 1_body/O/job_POSCAR_O_hollow (存在)
  3. 1_body/OH/job_POSCAR_OH_hollow (存在)
  4. 1_body/H_hollow/job_POSCAR_H_hollow (存在)
  5. 1_body/H_top/job_POSCAR_H_top (存在)
  6. 2_body/H_O/job_POSCAR_A_hollow_O__B_hollow_H (存在)
  7. 2_body/H/job_POSCAR_A_hollow_H__B_hollow_H (存在)
  8. 2_body/OH_H/job_POSCAR_A_hollow_OH__B_hollow_H (存在)
  9. 2_body/test_H2O/job_POSCAR (存在)
✔ 已删除: 1_body/H2/job_POSCAR_H2_top
✔ 已删除: 1_body/O/job_POSCAR_O_hollow
✔ 已删除: 1_body/OH/job_POSCAR_OH_hollow
✔ 已删除: 1_body/H_hollow/job_POSCAR_H_hollow
✔ 已删除: 1_body/H_top/job_POSCAR_H_top
✔ 已删除: 2_body/H_O/job_POSCAR_A_hollow_O__B_hollow_H
✔ 已删除: 2_body/H/job_POSCAR_A_hollow_H__B_hollow_H
✔ 已删除: 2_body/OH_H/job_POSCAR_A_hollow_OH__B_hollow_H
✔ 已删除: 2_body/test_H2O/job_POSCAR

完成！
  已删除: 9 个目录
  失败: 0 个目录
